<a href="https://colab.research.google.com/github/eddiecqw/Human-Image-Generator-with-Animal-Features/blob/main/ProgramMatcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
!pip install openai rapidfuzz pandas openpyxl requests -q


In [51]:
import os
import re
import pandas as pd
from typing import Optional, Dict, List, Tuple
from rapidfuzz import fuzz, process
import requests
from google.colab import userdata

# 1. 讀取 API Key
try:
    my_key = userdata.get('DEEPSEEK_API_KEY').strip()
    print(f"🔑 DeepSeek Key 讀取成功！長度: {len(my_key)} 位")
except Exception as e:
    print(f"❌ API Key 讀取失敗: {e}")

# 2. 掛載 Google Drive
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        print("✅ Google Drive 掛載成功！")
    else:
        print("💡 Google Drive 已處於掛載狀態。")
except Exception as e:
    print(f"⚠️ Google Drive 掛載異常: {e}")


🔑 DeepSeek Key 讀取成功！長度: 35 位
💡 Google Drive 已處於掛載狀態。


In [52]:
class DeepSeekProgramMatcher:
    def __init__(self, hesa_programs_list: List[str], api_key: str, model_name: str = "deepseek-chat"):
        self.raw_hesa_names = hesa_programs_list
        self.api_key = api_key
        self.model_name = model_name
        self.api_url = "https://api.deepseek.com/chat/completions"
        self.match_cache = {}

        self.std_names_dict = {}
        for name in hesa_programs_list:
            norm_key = self.normalize(name)
            if norm_key not in self.std_names_dict:
                self.std_names_dict[norm_key] = []
            self.std_names_dict[norm_key].append(name)

    def _pick_best_candidate(self, matched_key: str, raw_name: str) -> str:
        candidates = self.std_names_dict.get(matched_key, [])
        if not candidates: return ""
        if raw_name in candidates: return raw_name
        return min(candidates, key=len)

    # ==============================================================
    # 🌟 新增：結構化解析器 (精準分離 大學、學位、主修)
    # ==============================================================
    @staticmethod
    def _parse_structure(name: str) -> Tuple[str, str, str]:
        """將字串解析為 (大學, 學位層級歸類, 原始學位字串)"""
        if not isinstance(name, str): return "", "", ""
        parts = name.split('-', 2)
        if len(parts) >= 2:
            uni = parts[0].strip()
            level_raw = parts[1].strip().lower()

            # 學位硬性歸類，防止 BA/BScEcon 等細微差異導致誤殺，但嚴防跨階級
            level_type = 'other'
            if re.search(r'\b(msc|ma|msci|meng|mchem|mmath|mphys|llm)\b', level_raw):
                level_type = 'master'
            elif re.search(r'\b(bsc|ba|beng|bmus|llb|bscecon|bd)\b', level_raw):
                level_type = 'bachelor'

            return uni, level_type, level_raw
        return "", "", ""

    @staticmethod
    def normalize(name: str) -> str:
        if not isinstance(name, str) or pd.isna(name): return ""
        name_lower = name.lower().strip()

        # 1. 殺掉帶有學制關鍵字的括號及其內容
        name_lower = re.sub(r'[\(\[][^\)\]]*(placement|industry|abroad|foundation|ind|professional)[^\)\]]*[\)\]]', ' ', name_lower, flags=re.IGNORECASE)

        # 🌟 2. [新增解決方案] 殺掉包含合作學院、校區名稱的括號 (Franchise/Campus info)
        name_lower = re.sub(r'[\(\[][^\)\]]*(college|campus|centre)[^\)\]]*[\)\]]', ' ', name_lower, flags=re.IGNORECASE)

        # 3. 殺掉以 "with" 開頭的長尾學制後綴
        name_lower = re.sub(r'\bwith\s+(a\s+|an\s+)?(professional\s+|optional\s+|integrated\s+)?(placement|year|foundation|study|industry)\b.*', ' ', name_lower, flags=re.IGNORECASE)

        # 4. 殺掉直接標註的年份縮寫後綴
        name_lower = re.sub(r'\b(yr|year)\s+in\s+(ind|industry).*', ' ', name_lower, flags=re.IGNORECASE)

        noise_words = [r'\(hons\)', r'\bhons\b', r'\bft\b', r'\bpt\b', r'full time', r'part time']
        for noise in noise_words:
            name_lower = re.sub(noise, ' ', name_lower)

        name_lower = name_lower.replace('&', ' and ')
        name_lower = name_lower.replace('w/', ' with ')

        degree_map = {
            r'\bbsc\b': 'bachelor of science', r'\bba\b': 'bachelor of arts',
            r'\bmsc\b': 'master of science', r'\bmsci\b': 'master of science',
            r'\bma\b': 'master of arts', r'\bbeng\b': 'bachelor of engineering',
            r'\bmeng\b': 'master of engineering', r'\bmchem\b': 'master of chemistry',
            r'\bmmath\b': 'master of mathematics', r'\bmphys\b': 'master of physics',
            r'\bllb\b': 'bachelor of laws', r'\bllm\b': 'master of laws'
        }
        for pat, repl in degree_map.items():
            name_lower = re.sub(pat, repl, name_lower)

        name_lower = re.sub(r'[^\w\s]', ' ', name_lower)
        name_lower = re.sub(r'\s+', ' ', name_lower).strip()
        return name_lower

    def _call_deepseek_rerank(self, raw_name: str, candidates: List[str]) -> Optional[str]:
        if not self.api_key or not candidates: return None

        candidates_text = "\n".join([f"- {cand}" for cand in candidates])
        system_prompt = "You are an expert in UK Higher Education degree programs and course structures. Your task is to select the exact matching standard HESA program name for an input UCAS course."

        user_prompt = f"""Input Program Name: "{raw_name}"

Valid HESA Candidates List:
{candidates_text}

CRITICAL RULES FOR PROGRAM MATCHING:
1. PRIORITY 1: The University Name MUST MATCH perfectly.
2. PRIORITY 2: The primary subject MUST MATCH perfectly.
3. PRIORITY 3: Modifiers like 'with Placement' are the LOWEST priority.
4. TRADE-OFF: Always pick a candidate with the SAME University and SAME Major over a candidate with the wrong major but matching suffixes.
5. STRICT DEGREE LEVEL: Bachelor's MUST NEVER match Master's.
6. If no candidate shares BOTH the correct University and Core Major, output 'None'.

Your verbatim choice (or 'None'):"""

        headers = {"Content-Type": "application/json", "Authorization": f"Bearer {self.api_key}"}
        payload = {"model": self.model_name, "messages": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}], "temperature": 0.0, "max_tokens": 60}

        try:
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            reply = response.json()["choices"][0]["message"]["content"].strip().replace('"', '').replace("'", "")
            if reply.lower() == 'none' or reply not in candidates: return None
            return reply
        except Exception:
            return None

    def match(self, raw_name: str) -> Dict:
        if pd.isna(raw_name) or str(raw_name).strip() == "":
             return {'matched_name': None, 'method': 'none'}

        raw_name = str(raw_name).strip()
        if raw_name in self.match_cache: return self.match_cache[raw_name]

        normalized = self.normalize(raw_name)

        # 階段一：絕對一致
        if normalized in self.std_names_dict:
            result = {'matched_name': self._pick_best_candidate(normalized, raw_name), 'method': 'clean_exact'}
            self.match_cache[raw_name] = result
            return result

        # ==============================================================
        # 🌟 階段二改寫：建立「硬過濾局部池 (Local Pool)」
        # ==============================================================
        target_uni, target_level_type, target_level_raw = self._parse_structure(raw_name)

        local_pool = []
        if target_uni:
            # 1. 嚴格篩選同大學的科系
            uni_filtered = [name for name in self.raw_hesa_names if name.startswith(target_uni + "-")]

            # 2. 嚴格篩選同階級學位 (例如 BA 不能混到 MSc 裡面)
            for cand_name in uni_filtered:
                _, cand_level_type, cand_level_raw = self._parse_structure(cand_name)

                # 判斷：階級要一樣 (bachelor == bachelor)。
                # 特殊防呆：如果 UCAS 是 BA，但 HESA 是 BSc，這兩者都是 bachelor。
                # 我們把更細的判斷交給 LLM 去做，但絕不允許 BA 去跟 MA 做模糊匹配。
                if cand_level_type == target_level_type:
                    local_pool.append(cand_name)

            # 降級防呆 (Fallback)：萬一學位分類器誤判導致池子空了，退回「只要大學一樣就好」的池子
            if not local_pool and uni_filtered:
                local_pool = uni_filtered

        # 如果還是空的 (可能大學名字剛好有打錯字)，才退回全庫模糊搜索
        pool_to_search = local_pool if local_pool else self.raw_hesa_names

        # 階段三：在縮小範圍的「局部池」裡做模糊召回 (保障 Top 8)
        raw_candidates_set = process.extract(normalized, pool_to_search, scorer=fuzz.token_set_ratio, limit=8)
        raw_candidates_sort = process.extract(normalized, pool_to_search, scorer=fuzz.WRatio, limit=8)
        combined_candidates = list(set([c[0] for c in raw_candidates_set] + [c[0] for c in raw_candidates_sort]))

        # 階段四：DeepSeek 終審
        llm_choice = self._call_deepseek_rerank(raw_name, combined_candidates)

        if llm_choice:
            result = {'matched_name': llm_choice, 'method': 'deepseek_program_rerank'}
        else:
            result = {'matched_name': None, 'method': 'none'}

        self.match_cache[raw_name] = result
        return result

In [53]:
# -*- coding: utf-8 -*-
# ==========================================
# Cell 2: 文件讀取、數據清洗與匹配執行 (內建時間計時器)
# ==========================================
import time  # 🌟 引入時間模組

def read_data_file(filepath: str, sheet_name=0) -> pd.DataFrame:
    ext = filepath.lower()
    if ext.endswith('.csv'):
        try:
            return pd.read_csv(filepath, encoding='utf-8')
        except UnicodeDecodeError:
            return pd.read_csv(filepath, encoding='gbk')
    elif ext.endswith(('.xls', '.xlsx')):
        actual_sheet = sheet_name if sheet_name is not None else 0
        return pd.read_excel(filepath, sheet_name=actual_sheet)
    else:
        raise ValueError(f"不支援的檔案格式: {filepath}")

def process_program_files(
    hesa_filepath: str, ucas_filepath: str, output_filepath: str,
    api_key: str, hesa_sheet=0, ucas_sheet=0,
    hesa_col: str = "HESA Title", ucas_col: str = "UCAS Title"
):
    print("\n📥 正在讀取標準 HESA 科系庫...")
    hesa_df = read_data_file(hesa_filepath, sheet_name=hesa_sheet)
    ucas_df = read_data_file(ucas_filepath, sheet_name=ucas_sheet)

    for df, col, fname in [(hesa_df, hesa_col, "HESA"), (ucas_df, ucas_col, "UCAS")]:
        if col not in df.columns:
            print(f"❌ 錯誤：列名 '{col}' 在 {fname} 文件中不存在！可用的列名有：\n{list(df.columns)}")
            return

    # 提取 HESA 標準科系：去空、去重
    hesa_raw = hesa_df[hesa_col].dropna().astype(str).str.strip()
    hesa_clean = hesa_raw[hesa_raw != ""].unique().tolist()
    print(f"✅ HESA 標準科系：原始 {len(hesa_df)} 行，有效唯一科系 {len(hesa_clean)} 個")

    # 初始化配對大腦
    matcher = DeepSeekProgramMatcher(hesa_programs_list=hesa_clean, api_key=api_key)

    # 處理 UCAS 待匹配科系 (過濾空值)
    valid_mask = ucas_df[ucas_col].notna() & (ucas_df[ucas_col].astype(str).str.strip() != "")
    ucas_work = ucas_df.loc[valid_mask].copy()
    total_work_count = len(ucas_work)
    print(f"📋 UCAS 待匹配科系數：{total_work_count} (已自動過濾空行)")

    # 🌟 1. 計時器起點：在配對迴圈正式開始前記錄當前時間
    start_time = time.time()

    print(f"🚀 開始智能配對，共 {total_work_count} 條記錄 (呼叫 API 中)...")
    matched_results, methods = [], []

    for index, row in ucas_work.iterrows():
        target_name = row[ucas_col]

        # 每處理 50 筆列印一次進度
        if (len(matched_results) % 50 == 0 and len(matched_results) > 0):
            print(f"   進度: {len(matched_results)} / {total_work_count}")

        res = matcher.match(target_name)
        matched_results.append(res['matched_name'])
        methods.append(res['method'])

    # 🌟 2. 計時器終點：配對結束，計算時間差
    end_time = time.time()
    elapsed_time = end_time - start_time

    # 將秒數轉換為 分鐘 與 秒數
    mins, secs = divmod(elapsed_time, 60)

    # 寫回暫存區
    ucas_work['HESA_Matched_Program'] = matched_results
    ucas_work['Match_Method'] = methods

    # 將結果合併回原始結構
    output_df = ucas_df.copy()
    output_df['HESA_Matched_Program'] = None
    output_df['Match_Method'] = None
    output_df.loc[valid_mask, 'HESA_Matched_Program'] = ucas_work['HESA_Matched_Program'].values
    output_df.loc[valid_mask, 'Match_Method'] = ucas_work['Match_Method'].values

    # 核心過濾：只保留 3 欄
    output_df = output_df[[ucas_col, 'HESA_Matched_Program', 'Match_Method']]

    if output_filepath.lower().endswith('.csv'):
        output_df.to_csv(output_filepath, index=False, encoding='utf-8-sig')
    else:
        output_df.to_excel(output_filepath, index=False)

    print(f"\n🎉 配對完成！結果已保存至:\n👉 {output_filepath}")

    # 🌟 3. 列印精準的時間統計報告
    print("==================================================")
    print(f"⏱️  【配對時間統計報告】")
    print(f"    總計處理記錄: {total_work_count} 筆")
    print(f"    總共消耗時間: {int(mins)} 分 {secs:.2f} 秒")
    if total_work_count > 0:
        print(f"    平均每筆耗時: {(elapsed_time / total_work_count):.3f} 秒")
    print("==================================================")

In [54]:
# 主程式區
if __name__ == "__main__":
    try:
        raw_key = userdata.get('DEEPSEEK_API_KEY')
    except:
        raw_key = os.environ.get('DEEPSEEK_API_KEY', "")

    DEEPSEEK_API_KEY = raw_key.strip() if raw_key else input("請輸入 DeepSeek API Key: ").strip()

    # 🌟 基礎路徑
    BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/SocioBridge/Program_data/"

    # 檔案名稱
    # 這裡因為你提供的是合併過的文件，我們假設 HESA 和 UCAS 都在同一個檔案裡，只是列名不同
    FILE_NAME = "UCAS x HESA prog.xlsx"
    FILE_NAME1 = FILE_NAME
    HESA_FILE = BASE_DIR + FILE_NAME
    UCAS_FILE = BASE_DIR + FILE_NAME1
    OUTPUT_FILE = BASE_DIR + "Programs_DeepSeek_Matched.xlsx"

    # 🌟 核心參數：請確保這裡的 Column 名字與你 Excel 第一行的表頭一模一樣！
    HESA_SHEET = "Full"
    UCAS_SHEET = "Full"     # 指定要讀取的工作表
    HESA_COLUMN = "HESA Program Index"                  # 👈 替換為 HESA 科系的真實列名
    UCAS_COLUMN = "UCAS Program Index"                  # 👈 替換為 UCAS 科系的真實列名

    if os.path.exists(HESA_FILE):
        process_program_files(
            hesa_filepath=HESA_FILE,
            ucas_filepath=UCAS_FILE,
            output_filepath=OUTPUT_FILE,
            api_key=DEEPSEEK_API_KEY,
            hesa_sheet=HESA_SHEET,
            ucas_sheet=UCAS_SHEET,
            hesa_col=HESA_COLUMN,
            ucas_col=UCAS_COLUMN
        )
    else:
        print(f"⚠️ 找不到檔案！請檢查路徑: {HESA_FILE}")


📥 正在讀取標準 HESA 科系庫...
✅ HESA 標準科系：原始 34343 行，有效唯一科系 27218 個
📋 UCAS 待匹配科系數：13144 (已自動過濾空行)
🚀 開始智能配對，共 13144 條記錄 (呼叫 API 中)...
   進度: 50 / 13144
   進度: 100 / 13144
   進度: 150 / 13144
   進度: 200 / 13144
   進度: 250 / 13144
   進度: 300 / 13144
   進度: 350 / 13144
   進度: 400 / 13144
   進度: 450 / 13144
   進度: 500 / 13144
   進度: 550 / 13144
   進度: 600 / 13144
   進度: 650 / 13144
   進度: 700 / 13144
   進度: 750 / 13144
   進度: 800 / 13144
   進度: 850 / 13144
   進度: 900 / 13144
   進度: 950 / 13144
   進度: 1000 / 13144
   進度: 1050 / 13144
   進度: 1100 / 13144
   進度: 1150 / 13144
   進度: 1200 / 13144
   進度: 1250 / 13144
   進度: 1300 / 13144
   進度: 1350 / 13144
   進度: 1400 / 13144
   進度: 1450 / 13144
   進度: 1500 / 13144
   進度: 1550 / 13144
   進度: 1600 / 13144
   進度: 1650 / 13144
   進度: 1700 / 13144
   進度: 1750 / 13144
   進度: 1800 / 13144
   進度: 1850 / 13144
   進度: 1900 / 13144
   進度: 1950 / 13144
   進度: 2000 / 13144
   進度: 2050 / 13144
   進度: 2100 / 13144
   進度: 2150 / 13144
   進度: 2200 / 13144
   進度: 2250 / 1

In [72]:
# -*- coding: utf-8 -*-
# ==========================================
# Cell 3: 匹配結果質量審計與數據分析 (含互動式錯誤報表匯出)
# ==========================================
import pandas as pd
import os

def calculate_program_metrics():
    # 自動讀取 Cell 2 設定的輸出檔案路徑
    RESULT_FILE = BASE_DIR + "Programs_DeepSeek_Matched.xlsx"

    if not os.path.exists(RESULT_FILE):
        print("❌ 找不到比對結果文件，請先完整運行 Cell 2！")
        return

    df = pd.read_excel(RESULT_FILE)
    total_rows = len(df)

    # 讀取 Cell 2 設定的欄位名稱
    INPUT_COL = UCAS_COLUMN
    MATCHED_COL = 'HESA_Matched_Program'
    GROUND_TRUTH_COL = "Validated UCAS Index"

    problem_excel_rows = []
    audit_export_df = None # 準備用來裝錯配資料的 DataFrame

    print("==================================================")
    print("      📊 專業科系數據匹配審計與質量報告            ")
    print("==================================================")
    print(f"總處理科系數量: {total_rows} 條")

    matched_count = df[MATCHED_COL].notna().sum()
    none_count = df[MATCHED_COL].isna().sum()

    print(f"成功預測出實體: {matched_count} 條 ({(matched_count/total_rows)*100:.2f}%)")
    print(f"預測為 None (未能匹配): {none_count} 條 ({(none_count/total_rows)*100:.2f}%)")

    # --------------------------------------------------
    # 情況 A：有提供人工標準答案列 -> 進行深度分析
    # --------------------------------------------------
    if GROUND_TRUTH_COL and GROUND_TRUTH_COL in df.columns:

        true_none = (df[MATCHED_COL].isna() & df[GROUND_TRUTH_COL].isna()).sum()
        false_none = (df[MATCHED_COL].isna() & df[GROUND_TRUTH_COL].notna()).sum()

        print(f"   ➜ 其中「模型與人工皆為 None」(正確棄權): {true_none} 條")
        print(f"   ➜ 其中「模型為 None 但人工有答案」(錯配漏報): {false_none} 條")

        correct_matches = ((df[MATCHED_COL] == df[GROUND_TRUTH_COL]) | (df[MATCHED_COL].isna() & df[GROUND_TRUTH_COL].isna())).sum()
        accuracy = (correct_matches / total_rows) * 100
        print(f"\n🎯 最終科系匹配準確率 (Accuracy): {accuracy:.2f}%")

        # 抓出所有的錯配 (包含配錯，以及漏報的 false_none)
        mismatches = df[(df[MATCHED_COL] != df[GROUND_TRUTH_COL]) & ~(df[MATCHED_COL].isna() & df[GROUND_TRUTH_COL].isna())]

        # 收集行號並將資料存入匯出變數中
        problem_excel_rows = (mismatches.index + 2).tolist()
        audit_export_df = mismatches.copy()

        if len(mismatches) > 0:
            print(f"\n📈 【各比對方法錯誤分佈統計】")
            mismatches_copy = mismatches.copy()
            mismatches_copy['Match_Method'] = mismatches_copy['Match_Method'].fillna('未知方法(NaN)')

            method_counts = mismatches_copy['Match_Method'].value_counts()

            for method, count in method_counts.items():
                method_rows = (mismatches_copy[mismatches_copy['Match_Method'] == method].index + 2).tolist()
                print(f"   🔸 [{method}]: 共錯配 {count} 筆")
                print(f"      錯誤行號清單: {method_rows}")

            print(f"\n🚨 發現 {len(mismatches)} 個錯配項（前 20 條詳細如下）：")
            for idx, row in mismatches.head(20).iterrows():
                excel_row = idx + 2
                print(f"   - [行號 {excel_row}] UCAS 輸入: {row.get(INPUT_COL)} ")
                print(f"     大模型預測: {row[MATCHED_COL]}")
                print(f"     人工解答: {row[GROUND_TRUTH_COL]}")
                print(f"     比對方法: {row['Match_Method']}\n")

    # --------------------------------------------------
    # 情況 B：未提供人工標準答案列
    # --------------------------------------------------
    else:
        print("\n💡 提示: 未提供‘人工正確答案’列進行核對。以下為您展示前 20 條【未能匹配的科系(None)】：")
        unmatched_df = df[df[MATCHED_COL].isna()]

        problem_excel_rows = (unmatched_df.index + 2).tolist()
        audit_export_df = unmatched_df.copy()

        if len(unmatched_df) > 0:
            for idx, row in unmatched_df.head(20).iterrows():
                excel_row = idx + 2
                print(f"   ❌ 棄權配對 -> [行號 {excel_row}] UCAS 名稱: '{row.get(INPUT_COL)}'")
        else:
            print("   🌟 太棒了！所有 UCAS 科系都成功找到了對應的 HESA 科系。")

    print("==================================================")
    print("📌【最終行號匯總】所有需要人工審查的 Excel 行號清單：")
    if problem_excel_rows:
        print(f"總計共：{len(problem_excel_rows)} 行")
        print(problem_excel_rows)
        print("\n💡 提示：以上數字已自動加上標題行補正，與 Excel 左側的行號完全一致。")
    else:
        print("🎉 完美！沒有任何需要檢查的行號。")

    # ==================================================
    # 🌟 核心新增：互動式詢問是否儲存及自訂檔案名稱
    # ==================================================
    if audit_export_df is not None and not audit_export_df.empty:
        print("==================================================")
        save_choice = input("❓ 是否需要將所有錯配/未匹配紀錄匯出為 Excel 檔案？(y/n): ").strip().lower()

        if save_choice in ['y', 'yes']:
            filename = input("📝 請輸入要儲存的檔案名稱 (例如: error_report_v1.xlsx): ").strip()

            # 防呆機制：如果忘記打 .xlsx，程式自動補上
            if not filename.endswith('.xlsx'):
                filename += '.xlsx'

            export_path = BASE_DIR + filename

            # 在輸出的 Excel 第一欄插入 "Original_Excel_Row" (原始行號)，方便追蹤回原表
            audit_export_df.insert(0, 'Original_Excel_Row', audit_export_df.index + 2)
            audit_export_df.to_excel(export_path, index=False)

            print(f"\n📁 【錯誤分析表匯出成功】")
            print(f"已將紀錄匯出至:\n👉 {export_path}")
        else:
            print("\n🚫 已取消檔案匯出。")

    print("==================================================")

calculate_program_metrics()

      📊 專業科系數據匹配審計與質量報告            
總處理科系數量: 13144 條
成功預測出實體: 12664 條 (96.35%)
預測為 None (未能匹配): 480 條 (3.65%)
   ➜ 其中「模型與人工皆為 None」(正確棄權): 406 條
   ➜ 其中「模型為 None 但人工有答案」(錯配漏報): 74 條

🎯 最終科系匹配準確率 (Accuracy): 87.09%

📈 【各比對方法錯誤分佈統計】
   🔸 [deepseek_program_rerank]: 共錯配 1078 筆
      錯誤行號清單: [10530, 10531, 10532, 10533, 10535, 10536, 10537, 10539, 10541, 10542, 10552, 10553, 10568, 10569, 10572, 10574, 10575, 10576, 10577, 10578, 10579, 10580, 10581, 10582, 10583, 10584, 10585, 10586, 10587, 10588, 10589, 10590, 10595, 10596, 10597, 10600, 10602, 10603, 10606, 10609, 10610, 10611, 10614, 10615, 10616, 10617, 10618, 10619, 10625, 10627, 10628, 10629, 10630, 10631, 10632, 10633, 10634, 10635, 10644, 10645, 10646, 10647, 10654, 10655, 10667, 10668, 10669, 10672, 10676, 10680, 10681, 10682, 10684, 10685, 10686, 10687, 10688, 10689, 10690, 10691, 10692, 10775, 10776, 10779, 10780, 10781, 10786, 10787, 10790, 10791, 10794, 10795, 10798, 10799, 10802, 10803, 10804, 10805, 10806, 10807, 10808, 1080